# 05 — Autograd from scratch

*A walk through `tensor::autograd` from the ground up: tapes, gradients, chain rule, broadcast backward, and the contraction operator that powers a Linear layer.*

By the end of this notebook you will:

1. Be able to read `tensor::autograd` source and explain what every closure captures and why.
2. Know how `backward()` walks a tape in reverse and accumulates gradients per `Variable`.
3. Understand how Einstein-style label broadcast turns into a `reduce-over-collapsed-axes` step in the backward pass.
4. Recognise contraction as the operator whose own backward is another contraction.

> Prerequisite: `00_intro.ipynb` (named-axis tensors and four arithmetic operations).
> Reference: [ADR-0007](../docs/arc42/09-decisions/0007-adopt-autograd-as-first-class-subsystem.md).

In [ ]:
#pragma cling add_include_path("../include")
#include <iostream>
#include <tensor/autograd/autograd.hpp>
#include <tensor/core/core.hpp>

using namespace tensor::autograd;
using tensor::core::Axis;
using tensor::core::DynamicShape;
using tensor::core::DynamicTensor;
using tensor::core::Shape;
using tensor::core::Tensor;

## 1 — `Variable` and `Tape`

Every named-axis tensor that should participate in a backward pass is wrapped in a `Variable<T, N>`. A `Variable` holds:

- The forward **value** (a `tensor::core::Tensor<T, N>`).
- A `std::shared_ptr<GradAccum<T, N>>` holding the accumulated gradient — `nullptr` if the variable does not require grad.

Operators (`+ - *`, `exp`, `log`, `relu`, …) construct a fresh `Variable` for the output and **record a closure** into the thread-local `Tape`. The closure captures `shared_ptr`s to the input and output gradient accumulators; during `backward()`, the tape walks closures in reverse order and each one reads its output's grad and contributes to its inputs'.

That is the whole design. The rest of this notebook just instantiates it.

In [ ]:
Tape::current().clear();
Variable<double, 1> x(
    Tensor<double, 1>(Shape<1>{Axis{"i", 3}}, {1.0, 2.0, 3.0}),
    /*requires_grad=*/true);
std::cout << "x.value() = " << x.value() << std::endl;
std::cout << "x.requires_grad() = " << x.requires_grad() << std::endl;
std::cout << "Tape size before any op: " << Tape::current().size() << std::endl;

## 2 — Backward by hand: `sum(x * x)`

Take a simple loss: `L = Σ_i x_i²`. The analytical gradient is `∂L/∂x_i = 2 x_i`.

When we evaluate `loss = sum_all(x * x)`, three tape entries are recorded:

1. The `x * x` step records the multiplication backward: `dL/dx_i = dL/dprod_i · x_i + dL/dprod_i · x_i = 2 dL/dprod_i · x_i` (combined from the two contributions because both inputs are the same `x`).
2. `sum_all` records that `dL/dx_i = dL/dscalar` for all `i` (broadcast the scalar back).
3. `backward(loss)` seeds `dL/dscalar = 1` and walks entries in reverse.

End result: `x.grad() = 2 x = (2, 4, 6)`.

In [ ]:
auto loss = sum_all(x * x);
std::cout << "L = sum(x*x) = " << loss.value()[0] << std::endl;
std::cout << "Tape size after composing the graph: " << Tape::current().size() << std::endl;
backward(loss);
std::cout << "dL/dx = (" << x.grad()[0] << ", " << x.grad()[1] << ", " << x.grad()[2] << ")" << std::endl;
std::cout << "expected: 2 * x = (2, 4, 6)" << std::endl;

## 3 — Chain rule on a mixed graph: `sum((a + b) * c)`

Compose three primitives. Each step records its own closure; `backward()` walks them in reverse.

Analytical gradients:

- `∂L/∂a_i = c_i`, `∂L/∂b_i = c_i`, `∂L/∂c_i = (a + b)_i`.

In [ ]:
Tape::current().clear();
Variable<double, 1> a(Tensor<double, 1>(Shape<1>{Axis{"i", 2}}, {1.0, 2.0}), true);
Variable<double, 1> b(Tensor<double, 1>(Shape<1>{Axis{"i", 2}}, {3.0, 4.0}), true);
Variable<double, 1> c(Tensor<double, 1>(Shape<1>{Axis{"i", 2}}, {5.0, 6.0}), true);
auto loss3 = sum_all((a + b) * c);
std::cout << "L = " << loss3.value()[0] << " (expect 56)" << std::endl;
backward(loss3);
std::cout << "da = (" << a.grad()[0] << ", " << a.grad()[1] << ") (expect c = 5, 6)" << std::endl;
std::cout << "db = (" << b.grad()[0] << ", " << b.grad()[1] << ") (expect c = 5, 6)" << std::endl;
std::cout << "dc = (" << c.grad()[0] << ", " << c.grad()[1] << ") (expect a+b = 4, 6)" << std::endl;

## 4 — Activations with explicit derivatives

The activation primitives (`exp`, `log`, `relu`, `neg`) each record a closure that knows its own derivative:

| Primitive | Forward         | Backward                      |
| --------- | --------------- | ----------------------------- |
| `exp(x)`  | y = e^x         | dy/dx = y          (uses output) |
| `log(x)`  | y = log(x)      | dy/dx = 1/x        (uses input)  |
| `relu(x)` | y = max(0, x)   | dy/dx = I(x > 0)   (uses input)  |
| `neg(x)`  | y = -x          | dy/dx = -1                       |

Below we verify `exp` numerically using the central-difference gradient checker.

In [ ]:
Variable<double, 1> z(Tensor<double, 1>(Shape<1>{Axis{"i", 3}}, {0.2, 0.7, 1.1}), true);
bool gc = gradient_check(
    [](Variable<double, 1> const& v) { return sum_all(exp(v) * v); },
    z);
std::cout << "gradient_check(sum(exp(z) * z)): " << (gc ? "OK" : "MISMATCH") << std::endl;

## 5 — Broadcast backward as `reduce-over-collapsed-axes`

When `a_i + b_j` produces `c_{ij}`, the forward broadcasts each input across the other's axis. The backward direction reverses this: a gradient on `c` of rank 2 must be **reduced** (summed) over each axis that did not exist on the input.

Mathematically:

- `∂L/∂a_i = Σ_j ∂L/∂c_{ij}`
- `∂L/∂b_j = Σ_i ∂L/∂c_{ij}`

If `L = Σ_{ij} c_{ij}` then `∂L/∂c_{ij} = 1`, so:

- `∂L/∂a_i = j_extent` (sum of j_extent ones)
- `∂L/∂b_j = i_extent`

The `DynamicVariable` wrapper handles this; the closure captures the forward `BroadcastPlan` and calls `unbroadcast()` during the backward.

In [ ]:
Tape::current().clear();
DynamicVariable<double> ai(DynamicTensor<double>(DynamicShape{Axis{"i", 3}}, {1, 2, 3}), true);
DynamicVariable<double> bj(DynamicTensor<double>(DynamicShape{Axis{"j", 2}}, {10, 20}), true);
auto cij = ai + bj;
std::cout << "c shape rank = " << cij.value().shape().rank()
          << " size = " << cij.value().size() << std::endl;
auto loss5 = sum_all(cij);
backward(loss5);
std::cout << "grad a (expect (2, 2, 2)): (" << ai.grad()[0] << ", " << ai.grad()[1] << ", " << ai.grad()[2] << ")" << std::endl;
std::cout << "grad b (expect (3, 3)): ("     << bj.grad()[0] << ", " << bj.grad()[1] << ")" << std::endl;

## 6 — Contraction: `dot` and the gradient of a Linear layer

A Linear layer is `y = W · x + b`. With named axes:

```
y_i = Σ_j W_{ij} x_j + b_i
```

`dot(W, x)` is the contraction; the sum is over the shared axis `j`. The autograd rule is:

- `∂L/∂W_{ij} = ∂L/∂y_i · x_j`
- `∂L/∂x_j   = Σ_i ∂L/∂y_i · W_{ij}`

**Both gradients are themselves contractions.** That symmetry is why the autograd implementation reuses the same `contract()` kernel for forward and backward.

In [ ]:
Tape::current().clear();
DynamicVariable<double> W(
    DynamicTensor<double>(DynamicShape{Axis{"i", 2}, Axis{"j", 3}}, {1, 2, 3, 4, 5, 6}),
    true);
DynamicVariable<double> xv(DynamicTensor<double>(DynamicShape{Axis{"j", 3}}, {10, 20, 30}), true);
auto y = dot(W, xv);
std::cout << "y = (" << y.value()[0] << ", " << y.value()[1] << ")  (expect 140, 320)" << std::endl;
auto loss6 = sum_all(y);
backward(loss6);
std::cout << "grad W = [";
for (std::size_t k = 0; k < W.grad().size(); ++k) std::cout << W.grad()[k] << (k+1 < W.grad().size() ? ", " : "");
std::cout << "]  (expect [10, 20, 30, 10, 20, 30])" << std::endl;
std::cout << "grad x = (" << xv.grad()[0] << ", " << xv.grad()[1] << ", " << xv.grad()[2] << ")  (expect 5, 7, 9)" << std::endl;

## What you can build now

Combining everything, the forward + backward of a 2-layer MLP looks like:

```cpp
auto h    = relu(dot(W1, x) + b1);
auto pred = dot(W2, h) + b2;
auto loss = sum_all((pred - y_true) * (pred - y_true));  // MSE
backward(loss);
// gradients are now on W1, b1, W2, b2 — feed them into SGD.
```

That is exactly what `07_mlp-on-toy.ipynb` does. See you there.